# HAVD v4 — Mobese YOLOv8 Eğitimi (Colab GPU)

**Drive yapisi (kullanicinin yukledigi):**
```
MyDrive/havd/
├── havd_v4_dataset.zip
└── best_v3.pt
```

**Zip ictindeki yapi:**
```
havd_v4_dataset/
├── istanbul-traffic-vehicles/train/{images, labels}
├── cleaned/cam10/{images, labels}
├── cleaned/cam11/{images, labels}
└── auto_labeled/{file1_night, file3_day, file4_evening}/{images, labels}
```

**Suresi:** Colab T4 GPU, 60 epoch ≈ 1.5-2 saat.

## 1. Setup — GPU + Drive + Ultralytics

In [ ]:
# GPU kontrol
!nvidia-smi | head -15

In [ ]:
# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Ultralytics + numpy uyumlu surumler
!pip install -q -U "ultralytics>=8.4.0"

## 2. Dataset zip'ini cek + ac

In [ ]:
# Dataset'i Colab disk'ine kopyala ve ac
!mkdir -p /content/dataset
!cp /content/drive/MyDrive/havd/havd_v4_dataset.zip /content/
!ls -lh /content/havd_v4_dataset.zip
!unzip -q -o /content/havd_v4_dataset.zip -d /content/dataset
print('---')
!ls /content/dataset/
print('---')
!ls /content/dataset/havd_v4_dataset/ 2>/dev/null || ls /content/dataset/

In [ ]:
# Dataset root'unu tespit et (zip ya 'havd_v4_dataset/' acar ya da direkt klasorler)
from pathlib import Path

if Path('/content/dataset/havd_v4_dataset').exists():
    DATASET_ROOT = Path('/content/dataset/havd_v4_dataset')
else:
    DATASET_ROOT = Path('/content/dataset')
print('DATASET_ROOT:', DATASET_ROOT)
print('Icerigi:')
for p in sorted(DATASET_ROOT.iterdir()):
    print(' ', p.name)

In [ ]:
# best_v3.pt'yi cek (warm-start icin)
!cp /content/drive/MyDrive/havd/best_v3.pt /content/best_v3.pt
!ls -lh /content/best_v3.pt

## 3. Path listeleri (train/valid/test) + data.yaml uret

In [ ]:
import random
from pathlib import Path

OUT = DATASET_ROOT / 'final_v4'
OUT.mkdir(parents=True, exist_ok=True)

# Tum image+label cifti olan path'leri topla
search_dirs = [
    ('istanbul-traffic-vehicles',
        DATASET_ROOT/'istanbul-traffic-vehicles/train/images',
        DATASET_ROOT/'istanbul-traffic-vehicles/train/labels'),
    ('cleaned/cam10',
        DATASET_ROOT/'cleaned/cam10/images',
        DATASET_ROOT/'cleaned/cam10/labels'),
    ('cleaned/cam11',
        DATASET_ROOT/'cleaned/cam11/images',
        DATASET_ROOT/'cleaned/cam11/labels'),
    ('auto_labeled/file1_night',
        DATASET_ROOT/'auto_labeled/file1_night/images',
        DATASET_ROOT/'auto_labeled/file1_night/labels'),
    ('auto_labeled/file3_day',
        DATASET_ROOT/'auto_labeled/file3_day/images',
        DATASET_ROOT/'auto_labeled/file3_day/labels'),
    ('auto_labeled/file4_evening',
        DATASET_ROOT/'auto_labeled/file4_evening/images',
        DATASET_ROOT/'auto_labeled/file4_evening/labels'),
]

images = []
for tag, img_dir, lbl_dir in search_dirs:
    if not img_dir.exists():
        print(f'  [UYARI] yok: {img_dir}')
        continue
    n = 0
    for img in img_dir.glob('*.jpg'):
        if (lbl_dir / f'{img.stem}.txt').exists():
            images.append(str(img.resolve()))
            n += 1
    print(f'  {tag}: {n}')

print(f'\nTOPLAM: {len(images)}')

# 70/20/10 split (seed=42, deterministik)
random.seed(42)
random.shuffle(images)
n = len(images)
tr_end = int(n * 0.70)
vl_end = int(n * 0.90)

(OUT/'train.txt').write_text('\n'.join(images[:tr_end]))
(OUT/'valid.txt').write_text('\n'.join(images[tr_end:vl_end]))
(OUT/'test.txt').write_text('\n'.join(images[vl_end:]))

yaml_text = f'''path: {OUT}
train: train.txt
val: valid.txt
test: test.txt
nc: 4
names: ['bus', 'car', 'motorcycle', 'truck']
'''
(OUT/'data.yaml').write_text(yaml_text)

print(f'\nSplit: {tr_end} train / {vl_end-tr_end} valid / {n-vl_end} test')
print(f'data.yaml: {OUT/"data.yaml"}')

In [ ]:
# data.yaml ozet
!cat /content/dataset/havd_v4_dataset/final_v4/data.yaml 2>/dev/null || cat /content/dataset/final_v4/data.yaml

## 4. EGITIM — warm-start best_v3.pt + agresif augmentation

**Parametreler:**
- `epochs=60` (early stop `patience=20`)
- `imgsz=640`, `batch=16` (T4 GPU sinir)
- `lr0=0.005` (warm-start icin default 0.01'in yarisi)
- Agresif online augmentation: mosaic + mixup + copy_paste + rotation + hsv

In [ ]:
from ultralytics import YOLO
import torch

print('CUDA:', torch.cuda.is_available(),
      '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

model = YOLO('/content/best_v3.pt')

results = model.train(
    data=str(OUT / 'data.yaml'),
    epochs=60,
    patience=20,
    imgsz=640,
    batch=16,
    lr0=0.005,
    lrf=0.01,
    warmup_epochs=3.0,
    optimizer='SGD',
    # Augmentation
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
    degrees=10.0,
    translate=0.15,
    scale=0.6,
    fliplr=0.5,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
    # Yonetim
    project='/content/runs',
    name='mobese_v4',
    save=True,
    plots=True,
    seed=42,
    verbose=True,
)

## 5. Test setinde son degerlendirme

In [ ]:
best = YOLO('/content/runs/mobese_v4/weights/best.pt')
metrics = best.val(
    data=str(OUT / 'data.yaml'),
    split='test',
    plots=True,
    save_json=True,
)
print('mAP50:    {:.4f}'.format(metrics.box.map50))
print('mAP50-95: {:.4f}'.format(metrics.box.map))
print('Per-class mAP50:')
for i, n in enumerate(['bus', 'car', 'motorcycle', 'truck']):
    print(f'  {n:<12}: {metrics.box.maps[i]:.4f}')

## 6. Sonuclari Drive'a kopyala (best.pt + tum plot'lar)

In [ ]:
import shutil
from pathlib import Path

src = Path('/content/runs/mobese_v4')
dst = Path('/content/drive/MyDrive/havd/mobese_v4_results')
dst.mkdir(parents=True, exist_ok=True)

for item in src.rglob('*'):
    if item.is_file():
        rel = item.relative_to(src)
        target = dst / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, target)

# best.pt'yi root'a da koy (kolay erisim)
shutil.copy2(src/'weights/best.pt', Path('/content/drive/MyDrive/havd/best_v4.pt'))

print(f'Kopyalandi: {dst}')
print(f'En iyi model: /content/drive/MyDrive/havd/best_v4.pt')

In [ ]:
# Drive'daki ciktlarin listesi
!ls -la /content/drive/MyDrive/havd/
print()
!ls -la /content/drive/MyDrive/havd/mobese_v4_results/
print()
!ls -la /content/drive/MyDrive/havd/mobese_v4_results/weights/ 2>/dev/null